# 02 · Componente supervisado: predicción de fallas APS
**Equipo 9 – APS Failure at Scania Trucks**

**Objetivo:** predecir si un camión presenta una falla en el sistema APS (`class = pos`) a partir de 170 lecturas anónimas de sensores.

**Retos** (ver notebook 01):
- Desbalance fuerte: 1,67 % de positivos.
- Muchos faltantes.
- Variables muy asimétricas y redundantes.
- Costo de error asimétrico: `Costo = 10·FP + 500·FN`.

**Protocolo de evaluación**
1. Todo el preprocesamiento (`src/preprocessing.build_preprocessor`) y el remuestreo viven **dentro de un pipeline**, así que se ajustan solo con los folds de entrenamiento.
2. Selección de modelo, estrategia de desbalance, hiperparámetros y **umbral de decisión** se hacen con **validación cruzada estratificada sobre el train**.
3. El **test oficial** (16 000 camiones) se usa **una sola vez**, al final, para reportar el desempeño.

In [ ]:
import json
import sys
import time
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import os

os.environ["PYTHONPATH"] = str(ROOT)  # para que los workers de joblib importen src/
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from src.data import histogram_groups, load_aps
from src.metrics import (
    COST_FN,
    COST_FP,
    aps_cost,
    best_threshold_by_cost,
    cost_curve,
    evaluate,
    plot_confusion,
    plot_pr_curves,
)
from src.preprocessing import preprocessor_steps
from xgboost import XGBClassifier

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
SEED = 42
sns.set_theme(style="whitegrid", context="notebook")


def savefig(name):
    plt.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight")


X, y = load_aps("train")
X_test, y_test = load_aps("test")  # no se usa hasta la sección 6
POS_WEIGHT = (y == 0).sum() / (y == 1).sum()
print(f"Train {X.shape}, positivos {y.mean():.2%} | ratio neg/pos = {POS_WEIGHT:.0f}")

## 1. Selección y justificación de algoritmos

| Modelo | Por qué es apropiado | Supuestos | Ventajas | Limitaciones en APS |
|---|---|---|---|---|
| **Regresión Logística** (L2) | Línea base lineal e interpretable. Su probabilidad está bien calibrada para ajustar el umbral de costo. | Relación lineal entre las variables y el log-odds. Poca multicolinealidad. | Rápida, estable, coeficientes interpretables, admite `class_weight`. | No capta interacciones ni no linealidades. Depende de la imputación, del log y del escalado. Sufre con variables correlacionadas, que se mitigan con el filtro de correlación. |
| **Random Forest** | Ensamble de árboles que capta relaciones no lineales entre sensores. Es robusto a outliers y a la escala. | Ninguno distribucional. Asume que las observaciones son independientes. | Poca sensibilidad a los hiperparámetros, importancia de variables, `class_weight`. | Más pesado de entrenar. Las probabilidades salen poco calibradas. Con un desbalance extremo los árboles tienden hacia la clase mayoritaria si no se reponderan. |
| **XGBoost** (gradient boosting) | Estado del arte en datos tabulares. Fue la familia ganadora en APS (IDA 2016). | Ninguno distribucional. | Corrige errores de forma secuencial, regularización, `scale_pos_weight`, manejo nativo de NaN, compatible con SHAP. | Muchos hiperparámetros. Riesgo de sobreajuste con pocos positivos (1 000), que se controla con CV y regularización. |

El preprocesamiento es el mismo para los tres, para que la comparación sea justa: mediana + indicadores de NaN, log1p, eliminación de variables constantes y correlacionadas, y escalado. Los árboles no necesitan escalado, pero tampoco les afecta.

In [ ]:
def make_model(name, weighted):
    if name == "LogReg":
        return LogisticRegression(max_iter=3000, C=0.1, class_weight="balanced" if weighted else None)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=200, min_samples_leaf=2, max_features="sqrt", n_jobs=1,
                                      class_weight="balanced_subsample" if weighted else None, random_state=SEED)
    if name == "XGBoost":
        return XGBClassifier(n_estimators=300, learning_rate=0.08, max_depth=5, subsample=0.8,
                             colsample_bytree=0.6, tree_method="hist", n_jobs=1, eval_metric="aucpr",
                             scale_pos_weight=POS_WEIGHT if weighted else 1.0, random_state=SEED)
    raise ValueError(name)

def make_pipe(model_name, strategy):
    steps = preprocessor_steps()  # pasos planos: imblearn no admite pipelines anidados
    if strategy == "SMOTE":
        steps.append(("sampler", SMOTE(sampling_strategy=0.2, k_neighbors=5, random_state=SEED)))
    elif strategy == "Undersampling":
        steps.append(("sampler", RandomUnderSampler(sampling_strategy=0.1, random_state=SEED)))
    steps.append(("clf", make_model(model_name, weighted=(strategy == "Ponderación de clases"))))
    return ImbPipeline(steps).set_output(transform="pandas")

MODELS = ["LogReg", "RandomForest", "XGBoost"]
STRATEGIES = ["Sin tratamiento", "Ponderación de clases", "SMOTE", "Undersampling"]
cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)

## 2. Efecto del desbalance de clases
Para cada modelo se comparan cuatro estrategias:
- **Sin tratamiento**: sirve de referencia para medir el efecto del desbalance.
- **Ponderación de clases**: `class_weight` o `scale_pos_weight` = 59. Cada error en un positivo pesa 59 veces más en la función de pérdida.
- **SMOTE** (`sampling_strategy=0.2`): genera positivos sintéticos por interpolación **solo dentro del fold de entrenamiento**, hasta que haya 1 positivo por cada 5 negativos.
- **Undersampling aleatorio** (`sampling_strategy=0.1`): reduce los negativos hasta 10 por positivo. Entrena más rápido pero pierde información.

Métricas en CV estratificada de 5 folds: **PR-AUC** (principal, no depende del umbral y es informativa con desbalance), ROC-AUC, y Recall, Precision y F1 con el umbral por defecto de 0,5.

In [ ]:
rows = []
scoring = ["average_precision", "roc_auc", "recall", "precision", "f1"]
for m in MODELS:
    for s in STRATEGIES:
        t0 = time.time()
        r = cross_validate(make_pipe(m, s), X, y, cv=cv5, scoring=scoring, n_jobs=5)
        rows.append({"modelo": m, "estrategia": s,
                     **{k: r[f"test_{k}"].mean() for k in scoring},
                     "pr_auc_std": r["test_average_precision"].std(), "tiempo_s": time.time() - t0})
        print(f"{m:13s} | {s:22s} | PR-AUC={rows[-1]['average_precision']:.3f} | {rows[-1]['tiempo_s']:.0f}s")
imb = pd.DataFrame(rows).rename(columns={"average_precision": "pr_auc"})
imb.to_csv(RESULTS / "02_desbalance_cv.csv", index=False)

In [ ]:
display(imb.set_index(["modelo", "estrategia"]).drop(columns="tiempo_s").style.format("{:.3f}")
        .background_gradient(subset=["pr_auc", "recall"], cmap="Greens"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, metric, title in zip(axes, ["pr_auc", "recall", "precision"],
                             ["PR-AUC", "Recall (umbral 0.5)", "Precision (umbral 0.5)"]):
    sns.barplot(data=imb, x="modelo", y=metric, hue="estrategia", ax=ax, palette="Set2")
    ax.set(title=title, xlabel="", ylabel="", ylim=(0, 1)); ax.legend_.remove()
axes[0].legend(fontsize=8, loc="lower left")
plt.suptitle("Efecto de la estrategia de desbalance (CV 5-fold en train)", y=1.02)
plt.tight_layout(); savefig("02_efecto_desbalance"); plt.show()

**Lectura:**
- **XGBoost es el mejor modelo en todas las estrategias**: PR-AUC entre 0,853 y 0,893. Le siguen Random Forest (0,822–0,876) y la Regresión Logística (0,768–0,814). Que los modelos no lineales ganen confirma que la relación entre sensores y falla no es lineal.
- **En PR-AUC, el mejor resultado de cada modelo es *sin tratamiento*** (LogReg 0,814, RF 0,876, XGBoost 0,893). Ponderar clases o usar SMOTE **no mejora el ordenamiento** de los camiones por riesgo, que es lo que mide la PR-AUC. Lo que hacen es **desplazar las probabilidades hacia arriba**: con el umbral de 0,5, el Recall sube (LogReg pasa de 0,64 a 0,94 con ponderación) y la Precision cae (de 0,83 a 0,31). En la práctica equivalen a bajar el umbral.
- El **undersampling es la peor opción** en los modelos de árboles (XGBoost 0,853 frente a 0,893), porque descarta cerca del 85 % de los negativos y pierde información. Su única ventaja es la velocidad: Random Forest tarda 55 s en lugar de 210 s.
- **Decisión:** se entrena *sin remuestreo* y el desbalance se trata **ajustando de forma explícita el umbral de decisión según el costo** (sección 4). Es más transparente que SMOTE, no crea camiones sintéticos y optimiza directamente la métrica del negocio.

## 3. Ajuste de hiperparámetros
Para cada modelo se toma su mejor estrategia de desbalance (máxima PR-AUC en la sección 2) y se ejecuta un `RandomizedSearchCV` acotado: CV estratificada de 3 folds y scoring `average_precision`, para mantener un costo computacional razonable.

In [ ]:
best_strategy = imb.loc[imb.groupby("modelo")["pr_auc"].idxmax()].set_index("modelo")["estrategia"].to_dict()
print("Mejor estrategia por modelo:", best_strategy)

param_dists = {
    "LogReg": {"clf__C": loguniform(1e-3, 10)},
    "RandomForest": {"clf__n_estimators": randint(200, 500), "clf__max_depth": [None, 15, 25],
                     "clf__min_samples_leaf": randint(1, 6), "clf__max_features": ["sqrt", 0.2, 0.35]},
    "XGBoost": {"clf__n_estimators": randint(200, 700), "clf__learning_rate": loguniform(0.02, 0.2),
                "clf__max_depth": randint(3, 8), "clf__min_child_weight": randint(1, 8),
                "clf__subsample": uniform(0.6, 0.4), "clf__colsample_bytree": uniform(0.4, 0.5),
                "clf__reg_lambda": loguniform(0.5, 20)},
}
n_iter = {"LogReg": 8, "RandomForest": 8, "XGBoost": 15}
cv3 = StratifiedKFold(3, shuffle=True, random_state=SEED)
tuned, search_rows = {}, []
for m in MODELS:
    t0 = time.time()
    rs = RandomizedSearchCV(make_pipe(m, best_strategy[m]), param_dists[m], n_iter=n_iter[m],
                            scoring="average_precision", cv=cv3, n_jobs=-1, random_state=SEED, refit=False)
    rs.fit(X, y)
    best = make_pipe(m, best_strategy[m]).set_params(**rs.best_params_)
    tuned[m] = best
    search_rows.append({"modelo": m, "estrategia": best_strategy[m], "PR-AUC CV3": rs.best_score_,
                        "mejores parámetros": {k.replace("clf__", ""): (round(v, 4) if isinstance(v, float) else v)
                                               for k, v in rs.best_params_.items()},
                        "tiempo_s": time.time() - t0})
    print(f"{m}: PR-AUC={rs.best_score_:.3f} en {time.time()-t0:.0f}s → {search_rows[-1]['mejores parámetros']}")
search_df = pd.DataFrame(search_rows).set_index("modelo")
search_df

## 4. Umbral de decisión según el costo de los errores
El umbral de 0,5 supone que un FP y un FN cuestan lo mismo. En APS un **FN cuesta 50 veces más** que un FP, así que el umbral óptimo debería ser mucho más bajo. El umbral se elige con las **probabilidades out-of-fold** (CV de 5 folds en train): cada camión se predice con un modelo que no lo vio, y el umbral elegido es el que minimiza `10·FP + 500·FN`. El test no interviene en esta decisión.

In [ ]:
oof, thr_rows = {}, []
for m in MODELS:
    t0 = time.time()
    oof[m] = cross_val_predict(tuned[m], X, y, cv=cv5, method="predict_proba", n_jobs=5)[:, 1]
    thr, cost = best_threshold_by_cost(y, oof[m])
    e05, eopt = evaluate(y, oof[m], 0.5), evaluate(y, oof[m], thr)
    thr_rows.append({"modelo": m, "PR-AUC OOF": e05["pr_auc"], "costo @0.5": e05["cost"],
                     "umbral óptimo": thr, "costo @óptimo": cost,
                     "recall @óptimo": eopt["recall"], "precision @óptimo": eopt["precision"]})
    print(f"{m}: {time.time()-t0:.0f}s")
thr_df = pd.DataFrame(thr_rows).set_index("modelo")
thresholds = thr_df["umbral óptimo"].to_dict()
display(thr_df.style.format({"PR-AUC OOF": "{:.3f}", "umbral óptimo": "{:.3f}",
                             "recall @óptimo": "{:.3f}", "precision @óptimo": "{:.3f}",
                             "costo @0.5": "{:,.0f}", "costo @óptimo": "{:,.0f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for m in MODELS:
    t, c = cost_curve(y, oof[m])
    axes[0].plot(t, c, label=m)
    axes[0].scatter([thresholds[m]], [thr_df.loc[m, "costo @óptimo"]], zorder=5)
axes[0].set(xscale="log", xlabel="umbral de decisión (escala log)", ylabel="costo total en train (OOF)",
            title="Costo 10·FP + 500·FN vs umbral", ylim=(0, None))
axes[0].legend()
plot_pr_curves({m: (y, oof[m]) for m in MODELS}, ax=axes[1], title="Curvas Precision-Recall (OOF train)")
plt.tight_layout(); savefig("02_costo_vs_umbral"); plt.show()

# Trade-off precision/recall vs umbral para el mejor modelo OOF
final_name = thr_df["costo @óptimo"].idxmin()
print("Modelo con menor costo OOF (modelo final):", final_name)
ts = np.linspace(0.005, 0.95, 200)
pr = pd.DataFrame([evaluate(y, oof[final_name], t) for t in ts])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ts, pr["recall"], label="Recall"); ax.plot(ts, pr["precision"], label="Precision"); ax.plot(ts, pr["f1"], label="F1")
ax.axvline(thresholds[final_name], ls="--", c="k", lw=1, label=f"umbral de costo = {thresholds[final_name]:.3f}")
ax.axvline(0.5, ls=":", c="grey", lw=1, label="0.5")
ax.set(xlabel="umbral", title=f"Trade-off Precision/Recall – {final_name} (OOF)"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("02_tradeoff_umbral"); plt.show()

## 5. Entrenamiento final
Cada modelo ajustado se reentrena con **todo el train** (60 000 camiones) usando su umbral de costo. El **modelo final** se elige por el **menor costo OOF en train**, no por el resultado en test.

In [ ]:
fitted = {}
for m in MODELS:
    t0 = time.time()
    fitted[m] = clone(tuned[m]).fit(X, y)
    print(f"{m} entrenado en {time.time()-t0:.0f}s")

## 6. Evaluación en el test oficial (uso único)

In [ ]:
proba_test = {m: fitted[m].predict_proba(X_test)[:, 1] for m in MODELS}
rows = []
for m in MODELS:
    for label, t in [("0.5", 0.5), ("óptimo costo (OOF)", thresholds[m])]:
        rows.append({"modelo": m, "umbral": label, **evaluate(y_test, proba_test[m], t)})
test_df = pd.DataFrame(rows).set_index(["modelo", "umbral"])
test_df.to_csv(RESULTS / "02_test_metrics.csv")
display(test_df[["threshold", "recall", "precision", "f1", "pr_auc", "roc_auc", "fp", "fn", "cost"]]
        .style.format({"threshold": "{:.3f}", "recall": "{:.3f}", "precision": "{:.3f}", "f1": "{:.3f}",
                       "pr_auc": "{:.3f}", "roc_auc": "{:.3f}", "cost": "{:,.0f}"})
        .highlight_min(subset=["cost"], color="#c7e9c0"))

n_pos, n_neg = int(y_test.sum()), int((y_test == 0).sum())
refs = {"Siempre 'neg' (no hacer nada)": n_pos * COST_FN,
        "Siempre 'pos' (revisar todos)": n_neg * COST_FP,
        "Ganador IDA 2016 (referencia)": 9920}
print("Costos de referencia en test:", {k: f"{v:,}" for k, v in refs.items()})

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8.5))
for j, m in enumerate(MODELS):
    for i, (label, t) in enumerate([("umbral 0.5", 0.5), (f"umbral costo {thresholds[m]:.3f}", thresholds[m])]):
        yp = (proba_test[m] >= t).astype(int)
        plot_confusion(y_test, yp, ax=axes[i, j], title=f"{m} – {label}\ncosto = {aps_cost(y_test, yp):,}")
plt.tight_layout(); savefig("02_matrices_confusion_test"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
plot_pr_curves({m: (y_test, proba_test[m]) for m in MODELS}, ax=axes[0], title="Curvas Precision-Recall (test)")
costs = test_df.xs("óptimo costo (OOF)", level="umbral")["cost"].to_dict()
costs05 = test_df.xs("0.5", level="umbral")["cost"].to_dict()
bars = pd.DataFrame({"umbral 0.5": costs05, "umbral de costo": costs})
bars.plot.bar(ax=axes[1], color=["#BBBBBB", "#4C72B0"], rot=0)
axes[1].axhline(9920, ls="--", c="#DD5555", label="Ganador IDA 2016 (9 920)")
axes[1].set(title="Costo total en test", ylabel="10·FP + 500·FN"); axes[1].legend()
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():,.0f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom", fontsize=8)
plt.tight_layout(); savefig("02_costo_test"); plt.show()

**Resultados en test:**
| Modelo | Umbral | Recall | Precision | F1 | PR-AUC | FP | FN | **Costo** |
|---|---|---|---|---|---|---|---|---|
| LogReg | 0,5 | 0,677 | 0,852 | 0,755 | 0,864 | 44 | 121 | 60 940 |
| LogReg | 0,029 | 0,944 | 0,465 | 0,623 | 0,864 | 407 | 21 | 14 570 |
| Random Forest | 0,5 | 0,725 | 0,941 | 0,819 | 0,925 | 17 | 103 | 51 670 |
| Random Forest | 0,039 | **0,976** | 0,435 | 0,602 | 0,925 | 475 | **9** | 9 250 |
| **XGBoost** | 0,5 | 0,779 | **0,957** | **0,859** | **0,936** | 13 | 83 | 41 630 |
| **XGBoost** | **0,005** | 0,973 | 0,465 | 0,629 | **0,936** | 420 | 10 | **9 200** |

- **XGBoost es el mejor modelo en test**: PR-AUC 0,936 y ROC-AUC 0,996. Esto confirma la elección hecha con los datos de train, porque también tuvo el menor costo OOF (38 790, frente a 40 730 de RF y 42 750 de LogReg). Random Forest queda prácticamente empatado en costo (9 250).
- **El umbral de costo es la decisión más importante del modelo.** Con 0,5, XGBoost deja sin detectar 83 de las 375 fallas (costo 41 630). Con el umbral de 0,005 elegido en OOF deja solo 10 sin detectar y el **costo baja un 78 %, a 9 200**. A cambio, la Precision cae de 0,96 a 0,46: por cada falla real se revisa aproximadamente un camión sano más. Con costos de 10 frente a 500 ese intercambio conviene.
- **Comparación con referencias:**
  - No hacer nada (todo `neg`): 187 500.
  - Revisar todos los camiones: 156 250.
  - Ganador del reto IDA 2016: 9 920.
  - **Nuestro modelo: 9 200**, un 95,1 % menos que no hacer nada y por debajo de la referencia del reto.
- El F1 *baja* con el umbral de costo (de 0,86 a 0,63). Esto muestra que **F1 no es la métrica adecuada para este negocio**, porque pondera FP y FN por igual.
- El test tiene más positivos que el train (2,34 % frente a 1,67 %), y aun así el umbral elegido en train se transfiere bien.

## 7. Interpretación: variables más relevantes
- **Importancia por permutación** (modelo final): mide cuánto cae la PR-AUC al permutar cada variable **original**, lo que permite comparar entre modelos. Se calcula con un **20 % de validación separado del train**, entrenando en el 80 % restante. El test no se usa.
- **SHAP** (XGBoost): descompone cada predicción en aportes por variable y muestra la dirección del efecto. Se calcula sobre una muestra de 3 000 camiones del test, solo con fines explicativos (el modelo ya está fijado).

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
t0 = time.time()
model_pi = clone(tuned[final_name]).fit(X_tr, y_tr)
pi = permutation_importance(model_pi, X_val, y_val, scoring="average_precision",
                            n_repeats=5, random_state=SEED, n_jobs=-1)
perm = pd.DataFrame({"importancia": pi.importances_mean, "std": pi.importances_std}, index=X.columns)
perm = perm.sort_values("importancia", ascending=False)
perm.to_csv(RESULTS / "02_permutation_importance.csv")
print(f"Permutación en {time.time()-t0:.0f}s")

fig, ax = plt.subplots(figsize=(8, 6))
top15 = perm.head(15)[::-1]
ax.barh(top15.index, top15["importancia"], xerr=top15["std"], color="#4C72B0", capsize=2)
ax.set(xlabel="caída de PR-AUC al permutar", title=f"Importancia por permutación – {final_name}")
plt.tight_layout(); savefig("02_permutation_importance"); plt.show()

In [ ]:
import shap
groups = histogram_groups(X.columns)
xgb_pipe = fitted["XGBoost"]
X_shap = X_test.sample(3000, random_state=SEED)
Z_shap = xgb_pipe[:-1].transform(X_shap)  # todos los pasos menos el modelo (los samplers no actúan en transform)
explainer = shap.TreeExplainer(xgb_pipe.named_steps["clf"])
sv = explainer(Z_shap)

plt.figure()
shap.summary_plot(sv.values, Z_shap, max_display=15, show=False)
plt.title("SHAP – XGBoost (top 15 variables)"); savefig("02_shap_summary"); plt.show()

def var_group(col):
    if col.startswith("missingindicator_"):
        return "Indicadores de NaN"
    base = col[:2]
    return f"Histograma {base}_*" if base in groups else "Variables simples"
shap_imp = pd.Series(np.abs(sv.values).mean(axis=0), index=Z_shap.columns)
by_group = shap_imp.groupby(shap_imp.index.map(var_group)).sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.8))
(by_group / by_group.sum() * 100)[::-1].plot.barh(ax=ax, color="#55A868")
ax.set(xlabel="% de la importancia SHAP total", title="Importancia agregada por tipo de variable")
plt.tight_layout(); savefig("02_shap_por_grupo"); plt.show()
print(shap_imp.sort_values(ascending=False).head(10).round(3))

**Interpretación:**
- **Variables más relevantes.** Ambos métodos coinciden: `aa_000` y `ck_000` son las más importantes con mucha diferencia. En permutación, la PR-AUC cae 0,035 y 0,030 al permutarlas. Les siguen `ay_006`, `ay_008`, `al_000`, `bs_000`, `cb_000` y `ai_000`. Estas variables también tenían AUC univariado ≥ 0,96 en el EDA.
- **Dirección del efecto (SHAP).**
  - **Valores altos** de `aa_000`, `ck_000`, `bs_000`, `ai_000`, `bk_000` y `al_000` **aumentan** el riesgo de falla APS. Encaja con el EDA, donde los camiones con falla tenían mayor uso acumulado.
  - Valores altos de `cb_000`, `ag_009`, `ba_008` y `az_006` lo **reducen**.
- **Histogramas.** En conjunto suman cerca del **36 % de la importancia SHAP**: `ay` 7 %, `az` 6,5 %, `ag` 6 %, `cs` 5 %, `ee` 4 %, `ba` y `cn` 4 % cada uno. Importa sobre todo la **forma** de la distribución y no solo su total. Por ejemplo, en `ay` un peso alto en el bin 6 baja el riesgo y uno alto en el bin 8 lo sube. Coincide con los perfiles del notebook 01.
- **Indicadores de NaN.** Aportan apenas un 1 % en XGBoost. El faltante es informativo, pero esa señal queda recogida por las variables imputadas y por el resto de sensores, porque los faltantes aparecen en bloques correlacionados.
- **Limitación.** Las variables están **anonimizadas**. Se puede decir *qué sensores* y *en qué dirección* importan, pero no su significado físico. Para convertir esto en reglas de taller haría falta el diccionario de variables de Scania.

## 8. Costo de los errores y estrategia de mantenimiento preventivo

In [ ]:
p_final = proba_test[final_name]
t = thresholds[final_name]
edges = [0.0, t / 4, t, (1 + t) / 2, 1.0 + 1e-9]
names = ["Normal", "Vigilancia", "Revisión programada", "Taller inmediato"]
tier = pd.cut(p_final, edges, labels=names, right=False)
tiers = pd.DataFrame({"nivel": tier, "falla": y_test.values}).groupby("nivel", observed=False).agg(
    camiones=("falla", "size"), fallas_reales=("falla", "sum"))
tiers["tasa de falla"] = tiers["fallas_reales"] / tiers["camiones"]
tiers["% de todas las fallas"] = tiers["fallas_reales"] / tiers["fallas_reales"].sum()
tiers.index = [f"{nm}  [{lo:.3f}, {hi:.3f})" for nm, lo, hi in zip(names, edges[:-1], edges[1:])]
display(tiers.style.format({"tasa de falla": "{:.1%}", "% de todas las fallas": "{:.1%}"}))

ef = evaluate(y_test, p_final, t)
print(f"Modelo final {final_name} @ {t:.3f}: se envían {ef['tp'] + ef['fp']:,} camiones a revisión "
      f"({(ef['tp'] + ef['fp'])/len(y_test):.1%} de la flota); se detectan {ef['tp']}/{n_pos} fallas "
      f"(recall {ef['recall']:.1%}); quedan {ef['fn']} fallas sin detectar.")
base_cost = refs["Siempre 'neg' (no hacer nada)"]
print(f"Costo: {COST_FP}·{ef['fp']} + {COST_FN}·{ef['fn']} = {ef['cost']:,} "
      f"(vs {base_cost:,} sin modelo → ahorro {1 - ef['cost'] / base_cost:.1%})")

**Discusión de costos y propuestas de mantenimiento:**
**Costos.** Un FN, una falla APS no detectada, cuesta **50 veces** más que un FP, una revisión innecesaria: el camión se avería en ruta, hay que remolcarlo, se pierde la entrega y hay riesgo de seguridad por el sistema de frenos neumáticos. Por eso el umbral óptimo es muy bajo (0,005 en XGBoost): frente al umbral de 0,5, conviene revisar unos 400 camiones sanos más (407 FP adicionales) a cambio de evitar 73 fallas en ruta (de 83 FN a 10). Con ese umbral, el modelo envía a revisión **785 de 16 000 camiones (4,9 % de la flota)** y detecta **365 de 375 fallas (97,3 %)**.

**Propuestas de mantenimiento preventivo**
1. **Priorización por niveles de riesgo** (tabla de arriba), en lugar de una decisión binaria:
   - *Taller inmediato*: probabilidad alta. Concentra la mayoría de las fallas con una tasa de falla muy alta.
   - *Revisión programada*: probabilidad por encima del umbral de costo. Revisar el APS en la siguiente parada planificada.
   - *Vigilancia*: probabilidad entre ¼ del umbral y el umbral. Más frecuencia de lectura de sensores y reevaluar el modelo.
   - *Normal*: mantenimiento estándar.
2. **Monitoreo de los sensores clave** (`aa_000`, `ck_000`, `bs_000`, `ai_000` y la forma de los histogramas `ay`, `az`, `ag`): alertas cuando superen los percentiles observados en camiones con falla.
3. **Uso acumulado como disparador.** Los camiones con mayor uso (suma de bins de los histogramas unas 15 veces mayor en los camiones con falla) deberían tener intervalos de inspección del APS más cortos.
4. **Ajuste periódico del umbral.** El umbral de 0,005 depende de la calibración del modelo y de los costos. Si cambian los costos de taller, la flota o la tasa de fallas (en el test ya es 2,34 % frente a 1,67 %), hay que recalcularlo con datos recientes. La curva de costo de XGBoost es plana entre 0,002 y 0,02, así que el umbral es razonablemente robusto.
5. **Combinarlo con el componente no supervisado**: las anomalías detectadas sin etiqueta pueden revelar modos de falla nuevos que el modelo supervisado no conoce.

In [ ]:
summary = {
    "modelo_final": final_name,
    "mejor_estrategia_desbalance": best_strategy,
    "hiperparametros": search_df["mejores parámetros"].to_dict(),
    "umbral_costo_oof": thresholds,
    "costo_oof": thr_df["costo @óptimo"].to_dict(),
    "test": {f"{m}|{u}": {k: (int(v) if k in ("tn", "fp", "fn", "tp", "cost") else float(v))
                          for k, v in r.items()} for (m, u), r in test_df.iterrows()},
    "referencias_costo_test": refs,
}
with open(RESULTS / "02_supervisado_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False, default=str)
print(json.dumps(summary["test"][f"{final_name}|óptimo costo (OOF)"], indent=1))

## 9. Conclusiones del componente supervisado
1. **¿Se pueden predecir las fallas APS?** Sí. **XGBoost** con preprocesamiento compartido (mediana + indicadores, log1p, filtro de correlación) y **umbral ajustado por costo** detecta el **97,3 % de las fallas** del test oficial: Recall 0,973, PR-AUC 0,936, ROC-AUC 0,996. Su costo es **9 200**, por debajo de la referencia del reto IDA 2016 (9 920) y un 95 % menor que no hacer mantenimiento predictivo.
2. **Modelos.** XGBoost > Random Forest > Regresión Logística en PR-AUC, tanto en CV como en test. La logística sirve como base interpretable, pero sus fronteras lineales dejan escapar el doble de fallas a igual nivel de revisión (21 FN frente a 10).
3. **Desbalance.** Ponderar clases o usar SMOTE no mejoró la capacidad de ordenamiento (PR-AUC), y el undersampling la empeoró. Ajustar el **umbral según el costo** fue la estrategia más efectiva y transparente: redujo el costo un 78 % frente al umbral de 0,5.
4. **Métricas.** La accuracy (98 %+ en todos los casos) y el F1 son engañosos en este problema. Las métricas que deben guiar la decisión son PR-AUC, Recall y el **costo esperado**.
5. **Variables.** Unos pocos sensores (`aa_000`, `ck_000`, `bs_000`, `cb_000`, `ai_000`) y la forma de los histogramas `ay`, `az` y `ag` concentran la señal. La interpretación física queda limitada por la anonimización.
6. **Limitaciones.**
   - Los datos son un *snapshot* sin dimensión temporal, así que el modelo no dice *cuándo* fallará el camión.
   - El umbral óptimo es muy bajo y sensible a la calibración.
   - La búsqueda de hiperparámetros fue acotada (8–15 combinaciones) por costo computacional.
   - El test tiene una prevalencia distinta a la del train.

**Trabajo futuro:** calibrar las probabilidades (isotónica), probar LightGBM o CatBoost, usar validación temporal si hay fechas disponibles, y estimar el costo con intervalos (bootstrap) para medir su incertidumbre.